In [ ]:
import os
os._exit(0)

In [1]:
!git clone https://github.com/shyamaldutta897/dq_bpf.git

Cloning into 'dq_bpf'...
remote: Enumerating objects: 108, done.
remote: Counting objects: 100% (108/108), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 108 (delta 36), reused 95 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (108/108), 13.51 KiB | 3.38 MiB/s, done.
Resolving deltas: 100% (36/36), done.


In [1]:
import sys
sys.path.append('/content/dq_bpf')

In [3]:
%cd /content/dq_bpf/ 
!git pull

!ls /content/dq_bpf

/content/dq_bpf
Already up to date.
configs  dq_checks  dq_engine  pipelines  utils


In [4]:
from pyspark.sql.functions import *
from utils.spark_session import spark
spark

In [5]:
from google.colab import drive
drive.mount("/content/drive")
!ls /content/drive/MyDrive/dq_data/data/raw

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
yellow_tripdata_2026-01.parquet


In [5]:
trip_df=spark.read\
             .load('/content/drive/MyDrive/dq_data/data/raw')

In [7]:
import importlib
import dq_engine.engine as eng
importlib.reload(eng)
from dq_engine.engine import run_checks
import json 

file_path='/content/drive/MyDrive/dq_data/result.json'

result=run_checks(trip_df,"/content/dq_bpf/configs/user_rules.json")

with open(file_path,'w') as f:
    json.dump(result,f,indent=4)
    







In [35]:
import dq_engine.engine as eng
print(eng.run_checks.__code__.co_consts)

(None, ('not_null', 'negative_amount', 'num_check', 'double_check'), 'rules', 'type', 'columns')


In [27]:
import dq_engine.engine as eng

print(eng.run_checks)
print(eng.__file__)

<function run_checks at 0x795574fee0c0>
/content/dq_bpf/dq_engine/engine.py


In [8]:
import importlib
import dq_checks.double_check as dub
importlib.reload(dub)
print(dub.__file__)

/content/dq_bpf/dq_checks/double_check.py


In [14]:
!cat /content/dq_bpf/dq_checks/double_check.py

from pyspark.sql.functions import *
from pyspark.sql.types import *

def double_check(df,column):
    total=df.count()

    failed=df.filter((col(column).isNotNull())& (col(column).cast('double').isNull())).count()

    return{
        "field":column,
        "check":"Datatype - double",
        "total_rows":total,
        "failed_rows":failed,
        "percentage":failed/total
    }  

    